# SpendShield Synthetic Dataset Validation

This notebook validates the generated synthetic dataset and its manifest before any future baseline experiment.

## Objective

- Check schema, provenance, identifiers, amounts, timestamps, labels, and row-count reconciliation.
- Verify candidate-feature, audit-only, and forbidden-field separation.
- Verify non-overlapping temporal train/validation/test splits.
- Re-run generation with the recorded seed to check deterministic equivalence.

## Non-goals

No model training, anomaly detection, fraud classification, production inference, database access, or transaction decision is performed.

## Dataset boundary

The files under `data/synthetic/` are separate from MongoDB current state and Cassandra event history. They contain fictional records only. Synthetic scenario labels are not real fraud labels and cannot support real-world performance claims.

In [1]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "ml" / "synthetic_dataset_generator.py").exists()
)
sys.path.insert(0, str(REPOSITORY_ROOT))

import json

from ml.synthetic_dataset_generator import (
    SyntheticDatasetConfig,
    CANDIDATE_FEATURE_FIELDS,
    AUDIT_ONLY_FIELDS,
    FORBIDDEN_FEATURE_FIELDS,
    generate_dataset,
    validate_written_dataset,
)

OUTPUT_DIR = REPOSITORY_ROOT / "data" / "synthetic"
config_data = json.loads((OUTPUT_DIR / "generation_config.json").read_text(encoding="utf-8"))
CONFIG = SyntheticDatasetConfig(
    total_transactions=config_data["total_transactions"],
    user_count=config_data["user_count"],
    merchant_count=config_data["merchant_count"],
    duration_days=config_data["duration_days"],
    start_date=config_data["start_date"],
    currency=config_data["currency"],
    seed=config_data["seed"],
    dataset_version=config_data["dataset_version"],
    generator_version=config_data["generator_version"],
    scenario_counts=config_data["scenario_counts"],
)
manifest = json.loads((OUTPUT_DIR / "dataset_manifest.json").read_text(encoding="utf-8"))
label_dictionary = json.loads((OUTPUT_DIR / "label_dictionary.json").read_text(encoding="utf-8"))
{
    "dataset_version": manifest["dataset_version"],
    "generator_version": manifest["generator_version"],
    "label_notice": label_dictionary["synthetic_label_notice"],
}

{'dataset_version': 'v1',
 'generator_version': '1.0.0',
 'label_notice': 'Synthetic scenario labels identify intentionally generated scenarios. They do not represent confirmed fraud, real financial crime, or real-world risk outcomes.'}

In [2]:
validation = validate_written_dataset(OUTPUT_DIR)
assert validation["valid"], validation
validation

{'valid': True,
 'errors': [],
 'row_count': 10000,
 'column_count': 28,
 'user_count': 500,
 'account_count': 500,
 'merchant_count': 100,
 'category_count': 12,
 'scenario_distribution': {'normal': 7600,
  'synthetic_behavior_deviation': 300,
  'synthetic_combined_pattern': 100,
  'synthetic_high_amount': 800,
  'synthetic_rapid_repeat': 600,
  'synthetic_unusual_time': 600},
 'split_counts': {'train': 7061, 'validation': 1481, 'test': 1458}}

## Leakage boundary check

In [3]:
leakage_check = {
    "candidate_features": list(CANDIDATE_FEATURE_FIELDS),
    "audit_only_fields": list(AUDIT_ONLY_FIELDS),
    "forbidden_features": list(FORBIDDEN_FEATURE_FIELDS),
    "candidate_overlaps_audit": bool(set(CANDIDATE_FEATURE_FIELDS) & set(AUDIT_ONLY_FIELDS)),
    "candidate_overlaps_forbidden": bool(set(CANDIDATE_FEATURE_FIELDS) & set(FORBIDDEN_FEATURE_FIELDS)),
    "scenario_label_used_as_feature": "scenario_label" in CANDIDATE_FEATURE_FIELDS,
}
assert not leakage_check["candidate_overlaps_audit"]
assert not leakage_check["candidate_overlaps_forbidden"]
assert not leakage_check["scenario_label_used_as_feature"]
leakage_check

{'candidate_features': ['amount',
  'currency',
  'transaction_hour',
  'day_of_week',
  'merchant_category',
  'transaction_channel',
  'user_historical_transaction_count_before',
  'user_historical_average_amount_before',
  'time_since_previous_transaction_seconds',
  'user_historical_category_frequency_before'],
 'audit_only_fields': ['scenario_label',
  'scenario_id',
  'is_synthetic_scenario',
  'label_source',
  'label_definition',
  'dataset_type',
  'synthetic',
  'source_system',
  'dataset_version',
  'dataset_split',
  'scenario_injection_reason',
  'generator_rule',
  'synthetic_transaction_id',
  'synthetic_user_id',
  'synthetic_account_id',
  'synthetic_merchant_id'],
 'forbidden_features': ['future_transaction_count',
  'future_average_amount',
  'final_scenario_label',
  'scenario_injection_reason',
  'post_event_outcome',
  'manually_assigned_research_label',
  'confirmed_fraud',
  'fraud'],
 'candidate_overlaps_audit': False,
 'candidate_overlaps_forbidden': False,
 

## Reproducibility check

The same configuration and seed are run twice in memory. The generated rows and split assignments must be identical. The manifest creation timestamp is intentionally run metadata and may differ between executions.

In [4]:
first = generate_dataset(CONFIG)
second = generate_dataset(CONFIG)
reproducibility = {
    "same_records": first.records == second.records,
    "same_splits": first.splits == second.splits,
    "same_manifest_core": all(
        first.manifest[key] == second.manifest[key]
        for key in ("row_count", "column_count", "scenario_distribution", "split_counts", "random_seed")
    ),
}
assert all(reproducibility.values()), reproducibility
reproducibility

{'same_records': True, 'same_splits': True, 'same_manifest_core': True}

## Validation conclusion

The dataset is structurally valid, provenance-marked, leakage-aware, temporally split, and reproducible for controlled research. It is not real financial data, does not contain confirmed fraud labels, and is not approved for production decisions. Model training remains a separate future phase.